# ASAP8 VIP somatic voltage characterization

Single-session analysis of ASAP8+ VIP interneurons during passive Detection of Change.

The notebook is organized around the opening electrophysiology section of the lab meeting:

1. visually inspect somatic dF/F and recorded ROIs;
2. detect approximate spike times directly from dF/F;
3. separate high-confidence singletons from doublets and bursts;
4. compare waveform, bursting, plateau, and synchrony phenotypes across depth.

All paths and metadata are resolved through `VIPSessionRegistry`, the selected asset, and `asset.qc_dir`. Tables retain `session_id`, `dmd`, and `roi` so longitudinal ROI identities can be added later.


## 0. Imports and plotting style

The figure palette draws selectively from the PNWColors **Shuksan2** and **Sailboat** palettes: pastel blue/peach fills with darker outlines and summary lines.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import json

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import ndimage, signal
from IPython.display import display, HTML

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))

# PNWColors-inspired palette: primarily Shuksan2 and Sailboat.
NAVY = "#1D457F"
CHARCOAL = "#2D2926"
DARK_BLUE = "#3E5685"
DARK_PEACH = "#A8554E"
BLUE = "#5D74A5"
BLUE_FILL = "#B0CBE7"
PEACH = "#EBA07E"
PEACH_FILL = "#F2C5B4"
CREAM = "#FEF7C7"
TEAL = "#64A8A8"
GRAY = "#7A7A7A"
LIGHT_GRAY = "#D9D9D9"

DMD_COLORS = {1: BLUE, 2: PEACH}
DMD_FILLS = {1: BLUE_FILL, 2: PEACH_FILL}
DMD_DARK = {1: DARK_BLUE, 2: DARK_PEACH}

STTC_CMAP = LinearSegmentedColormap.from_list(
    "pnw_sttc",
    ["#F7F7F5", BLUE_FILL, BLUE, NAVY],
)

# Explicit z-ordering keeps annotations and event calls visible.
ZORDER = {
    "span": 0,
    "grid": 1,
    "trace": 2,
    "summary": 3,
    "marker": 4,
    "annotation": 5,
}

plt.rcParams.update({
    "figure.dpi": 115,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.transparent": False,
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
    "axes.axisbelow": True,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def save_panel(fig, name):
    if not SAVE_FIGURES:
        return
    for ext in ("png", "pdf", "svg"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", facecolor="white")


## 1. Select one session through the registry

In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")
TARGET_MOUSE = 852835
SESSION_IDX = -4  # None selects the latest registry session

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"

SAVE_FIGURES = True
SAVE_TABLES = True

registry = VIPSessionRegistry.from_basepath(BASE_PATH)
session_df = registry.sessions(
    subject_ids=[TARGET_MOUSE],
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
).sort_values("session_date").reset_index(drop=True)

assets = [registry.resolve_assets(row) for _, row in session_df.iterrows()]
asset = assets[SESSION_IDX]

show_cols = [c for c in [
    "session_id", "subject_id", "session_date", "session_type",
    "dmd1_depth", "dmd2_depth", "quality",
] if c in session_df.columns]

display(session_df[show_cols])
print("Selected:", asset.session_id)

## 2. Resolve paths and acquisition metadata from the asset

In [ ]:
VOLTAGE_QC_PATH = (
    asset.qc_dir
    / "voltage"
    / f"voltage_extraction_qc_{TRACE_VARIANT}.json"
)

with open(VOLTAGE_QC_PATH, "r") as f:
    voltage_qc = json.load(f)

FS = float(voltage_qc["sample_rate_hz"])
SUMMARY_PATH = Path(voltage_qc["summary_mat"])
TRACE_H5_PATH = (
    asset.derived_dir
    / "voltage"
    / f"voltage_session_traces_{TRACE_VARIANT}.h5"
)

OUT_DIR = asset.derived_dir / "asap8_somatic_ephys"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
for directory in (OUT_DIR, FIG_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

depths = {
    1: float(asset.metadata["dmd1_depth"]),
    2: float(asset.metadata["dmd2_depth"]),
}

print(f"Session:     {asset.session_id}")
print(f"Sample rate: {FS:,.3f} Hz")
print(f"Trace:       {TRACE_H5_PATH}")
print(f"Summary:     {SUMMARY_PATH}")
print(f"Depths:      DMD1={depths[1]:.0f} µm, DMD2={depths[2]:.0f} µm below pia")
print(f"Output:      {OUT_DIR}")

## 3. Trace and summary readers

In [ ]:
def _as_scalar(x):
    arr = np.asarray(x).squeeze()
    if arr.size == 1:
        value = arr.reshape(-1)[0]
        return value.item() if isinstance(value, np.generic) else value
    return arr


def _orient_trace_dataset(ds, expected_n_rois):
    arr = np.asarray(ds[()], dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D trace dataset, got {arr.shape}")
    if arr.shape[0] == expected_n_rois:
        return arr
    if arr.shape[1] == expected_n_rois:
        return arr.T
    raise ValueError(
        f"Neither axis matches expected_n_rois={expected_n_rois}; shape={arr.shape}"
    )


def _read_trial_timing(summary_path, n_trials, fs, trial_lengths):
    starts = None
    ends = None
    with h5py.File(summary_path, "r") as f:
        if "summary/trialTable/trialStartTimeInferred" in f:
            starts = np.asarray(
                f["summary/trialTable/trialStartTimeInferred"][()], dtype=float
            ).reshape(-1)[:n_trials]
        if "summary/trialTable/trialEndTimeFromPC" in f:
            ends = np.asarray(
                f["summary/trialTable/trialEndTimeFromPC"][()], dtype=float
            ).reshape(-1)[:n_trials]

    total = int(np.sum(trial_lengths))
    compressed_time_sec = np.arange(total, dtype=np.float64) / float(fs)
    acquisition_time_sec = np.empty(total, dtype=np.float64)
    trial_id = np.empty(total, dtype=np.int32)
    trial_slices = []

    cursor = 0
    for i, n in enumerate(trial_lengths):
        n = int(n)
        sl = slice(cursor, cursor + n)
        trial_id[sl] = i + 1
        trial_slices.append({"trial": i + 1, "start": cursor, "stop": cursor + n})
        if starts is not None and np.all(np.isfinite(starts)):
            start_sec = (starts[i] - starts[0]) * 86400.0
            acquisition_time_sec[sl] = start_sec + np.arange(n) / float(fs)
        else:
            acquisition_time_sec[sl] = compressed_time_sec[sl]
        cursor += n

    return {
        "compressed_time_sec": compressed_time_sec,
        "acquisition_time_sec": acquisition_time_sec,
        "trial_id": trial_id,
        "trial_slices": trial_slices,
        "trial_start_matlab_datenum": starts,
        "trial_end_matlab_datenum": ends,
    }

In [ ]:
def load_derived_voltage_traces(
    summary_path,
    trace_h5_path,
    fs,
    signal="dff",
):
    """Load processed voltage-session H5 data as DMD -> ROI x time."""

    with h5py.File(summary_path, "r") as sf:
        n_rois = np.asarray(
            sf["summary/nAnalysisROIs"][()],
            dtype=int,
        ).reshape(-1)

        trial_lengths = np.asarray(
            sf["summary/trialLineRanges/trialGlobalNLines"][()],
            dtype=int,
        ).reshape(-1)

    traces = {}

    with h5py.File(trace_h5_path, "r") as tf:
        for dmd, expected_n_rois in enumerate(n_rois, start=1):
            dataset_path = f"DMD{dmd}/{signal}"

            if dataset_path not in tf:
                raise KeyError(
                    f"{dataset_path!r} not found in {trace_h5_path}. "
                    f"Root groups: {list(tf.keys())}"
                )

            traces[dmd] = _orient_trace_dataset(
                tf[dataset_path],
                int(expected_n_rois),
            )

    sample_counts = {x.shape[1] for x in traces.values()}
    if len(sample_counts) != 1:
        raise ValueError(
            f"DMD datasets have different sample counts: {sample_counts}"
        )

    n_samples = sample_counts.pop()

    if trial_lengths.sum() != n_samples:
        raise ValueError(
            f"Summary describes {trial_lengths.sum():,} samples, "
            f"but the processed H5 contains {n_samples:,}."
        )

    timing = _read_trial_timing(
        summary_path,
        len(trial_lengths),
        fs,
        trial_lengths,
    )

    return traces, {
        "mode": "processed_trial_concatenated",
        "signal": signal,
        "trial_lengths_samples": trial_lengths,
        **timing,
    }

## 4. Reference images, ROI masks, and ROI manifest

In [ ]:
def orient_slap2_image_for_display(image):
    return np.flipud(np.asarray(image).T)


def orient_slap2_masks_for_display(masks):
    return np.asarray(masks).transpose(0, 2, 1)[:, ::-1, :]


def read_ref_image(summary_path, dmd):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/refIM"][dmd - 1, 0]
        image = np.asarray(f[ref][()], dtype=np.float32)
    return orient_slap2_image_for_display(image)


def read_roi_masks(summary_path, dmd):
    with h5py.File(summary_path, "r") as f:
        ref = f["summary/masks"][dmd - 1, 0]
        masks = np.asarray(f[ref][()], dtype=bool)
    return orient_slap2_masks_for_display(masks)


traces, trace_info = load_derived_voltage_traces(
    SUMMARY_PATH,
    TRACE_H5_PATH,
    FS,
    signal="dff",
)

segment_slices = (
    [slice(x["start"], x["stop"]) for x in trace_info["trial_slices"]]
    if trace_info["trial_slices"] is not None
    else [slice(0, traces[1].shape[1])]
)

duration_s = traces[1].shape[1] / FS
print(f"Trace mode: {trace_info['mode']}")
print(f"Duration:   {duration_s / 60:.2f} min")
for dmd, x in traces.items():
    print(f"DMD{dmd}: {x.shape[0]} ROIs x {x.shape[1]:,} samples")

reference_images = {dmd: read_ref_image(SUMMARY_PATH, dmd) for dmd in traces}
roi_masks = {dmd: read_roi_masks(SUMMARY_PATH, dmd) for dmd in traces}

manifest_rows = []
for dmd, masks in roi_masks.items():
    for roi, mask in enumerate(masks):
        yy, xx = np.nonzero(mask)
        manifest_rows.append({
            "subject_id": asset.subject_id,
            "session_id": asset.session_id,
            "dmd": dmd,
            "roi": roi,
            "depth_um": depths[dmd],
            "roi_area_px": int(mask.sum()),
            "centroid_x_px": float(np.mean(xx)),
            "centroid_y_px": float(np.mean(yy)),
        })

roi_manifest = pd.DataFrame(manifest_rows)
display(roi_manifest)

In [ ]:
fig, axes = plt.subplots(
    1,
    len(traces),
    figsize=(5.3 * len(traces), 4.7),
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    image = reference_images[dmd]
    masks = roi_masks[dmd]

    display_image = (
        image[:, :, 0]
        if image.ndim > 2
        else image
    )
    low, high = np.nanpercentile(
        display_image,
        [1, 99.7],
    )

    ax.imshow(
        display_image,
        cmap="gray",
        vmin=low,
        vmax=high,
        zorder=ZORDER["span"],
    )

    for roi, mask in enumerate(masks):
        ax.contour(
            mask,
            levels=[0.5],
            colors=[DMD_COLORS[dmd]],
            linewidths=1.6,
            zorder=ZORDER["summary"],
        )
        yy, xx = np.nonzero(mask)
        ax.text(
            np.mean(xx),
            np.mean(yy),
            str(roi),
            color="white",
            fontsize=9,
            ha="center",
            va="center",
            bbox={
                "facecolor": DMD_DARK[dmd],
                "edgecolor": "white",
                "linewidth": 0.5,
                "alpha": 0.88,
                "pad": 1.7,
            },
            zorder=ZORDER["annotation"],
        )

    ax.set_title(
        f"DMD{dmd} · {depths[dmd]:.0f} µm below pia"
    )
    ax.axis("off")

fig.suptitle(
    "ASAP8 somatic ROIs",
    color=NAVY,
    fontsize=18,
    y=0.98,
)
fig.tight_layout()
save_panel(fig, "01_reference_images_and_rois")
plt.show()


## 5. Visual inspection of the dF/F traces

The first plot compresses the complete recording for drift and activity-regime inspection. The second uses one editable time window for direct waveform comparison.

In [ ]:
OVERVIEW_BIN_MS = 2.0
ZOOM_SEC = (20.0, 25.0)

fig, axes = plt.subplots(
    len(traces),
    1,
    figsize=(
        13,
        2.5 + 1.0 * sum(x.shape[0] for x in traces.values()),
    ),
    sharex=True,
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    x = traces[dmd]
    step = max(
        1,
        int(round(OVERVIEW_BIN_MS / 1000 * FS)),
    )
    n_bins = x.shape[1] // step
    reduced = (
        x[:, :n_bins * step]
        .reshape(x.shape[0], n_bins, step)
        .mean(axis=2)
    )
    time_min = np.arange(n_bins) * step / FS / 60

    robust_range = (
        np.nanpercentile(reduced, 95, axis=1)
        - np.nanpercentile(reduced, 5, axis=1)
    )
    offsets = np.r_[
        0,
        np.cumsum(
            np.maximum(
                robust_range[:-1],
                np.nanmedian(robust_range),
            )
            * 3
        ),
    ]

    for roi, row in enumerate(reduced):
        centered = row - np.nanmedian(row)
        ax.plot(
            time_min,
            centered + offsets[roi],
            lw=0.60,
            color=DMD_COLORS[dmd],
            zorder=ZORDER["trace"],
        )
        ax.text(
            time_min[0] - 0.01 * time_min[-1],
            offsets[roi],
            f"ROI {roi}",
            ha="right",
            va="center",
            color=CHARCOAL,
            zorder=ZORDER["annotation"],
        )

    ax.set_yticks([])
    ax.set_ylabel(
        f"DMD{dmd}\n{depths[dmd]:.0f} µm"
    )
    ax.set_title(f"DMD{dmd}: full-session dF/F")

axes[-1].set_xlabel("Compressed session time (min)")
fig.tight_layout()
save_panel(fig, "02_full_session_trace_overview")
plt.show()


In [ ]:
t0, t1 = ZOOM_SEC
i0 = max(0, int(round(t0 * FS)))
i1 = min(traces[1].shape[1], int(round(t1 * FS)))

fig, axes = plt.subplots(
    len(traces),
    1,
    figsize=(
        13,
        2.3 + 0.90 * sum(x.shape[0] for x in traces.values()),
    ),
    sharex=True,
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    window = traces[dmd][:, i0:i1]
    time = np.arange(i0, i1) / FS
    scale = np.nanmedian(
        np.nanpercentile(window, 95, axis=1)
        - np.nanpercentile(window, 5, axis=1)
    )
    offsets = np.arange(window.shape[0]) * scale * 1.75

    for roi, row in enumerate(window):
        ax.plot(
            time,
            row - np.nanmedian(row) + offsets[roi],
            lw=0.75,
            color=DMD_COLORS[dmd],
            zorder=ZORDER["trace"],
        )
        ax.text(
            t0 - 0.012 * (t1 - t0),
            offsets[roi],
            f"ROI {roi}",
            ha="right",
            va="center",
            color=CHARCOAL,
            zorder=ZORDER["annotation"],
        )

    ax.set_yticks([])
    ax.set_ylabel(
        f"DMD{dmd}\n{depths[dmd]:.0f} µm"
    )
    ax.set_title(f"DMD{dmd}: {t0:g}–{t1:g} s")

axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
save_panel(fig, "03_zoomed_trace_comparison")
plt.show()


# Part I — Within-session electrophysiological phenotype

Spike timing and burst structure are estimated separately:

- **Candidate spikes** are local maxima detected directly on the full concatenated dF/F trace.
- **High-confidence singletons** are used for waveform measurements.
- Candidate peaks linked by short inter-peak intervals are classified as singletons, doublets, or bursts.
- A slow depolarization envelope is used only to estimate burst-window duration, not to decide how many spikes occurred.


## 6. Spike, burst, and waveform parameters

The candidate detector follows the visually interpretable `find_peaks` scheme tested on the example recording:

- height ≥ mean + 1.5 SD;
- prominence ≥ 0.10 dF/F;
- width ≥ 1 ms;
- peak distance ≥ 0.8 ms.

Waveform measurements require a higher 2-SD peak threshold and an isolated singleton. Bursts contain at least three candidate peaks linked by consecutive intervals of no more than 20 ms. Doublets remain a separate class.


In [ ]:
DETECTION = {
    # Approximate spike detection on the raw dF/F trace.
    "candidate_height_sd": 1.5,
    "waveform_height_sd": 2.0,
    "prominence_dff": 0.10,
    "prominence_window_ms": 50.0,
    "refractory_ms": 0.8,
    "min_width_ms": 1.0,

    # Spike-group taxonomy.
    "group_link_ms": 20.0,
    "burst_min_spikes": 3,

    # Independent slow-envelope estimate of burst duration.
    "burst_envelope_smooth_ms": 3.0,
    "burst_envelope_height_sd": 1.0,
    "burst_envelope_min_ms": 10.0,
    "burst_envelope_merge_gap_ms": 5.0,
    "burst_expand_pre_max_ms": 10.0,
    "burst_expand_post_max_ms": 50.0,

    # Isolated-spike waveforms.
    "waveform_pre_ms": 10.0,
    "waveform_post_ms": 20.0,
    "waveform_baseline_ms": (-3.5, -1.0),

    # Fixed-window burst examples.
    "burst_example_pre_ms": 10.0,
    "burst_example_post_ms": 100.0,

    # Simple preliminary plateau metric.
    "plateau_window_ms": (8.0, 50.0),
    "baseline_bin_s": 0.25,
    "baseline_percentile": 20,
    "baseline_smooth_s": 2.0,

    # Set >0 only if concatenation boundaries create visible artifacts.
    "exclude_trial_edge_ms": 0.0,
}

DETECTION


## 7. Whole-trace detector and phenotype helpers

Detection is performed once on each full concatenated ROI trace. The slow baseline used for the plateau metric is not used to find peaks.


In [ ]:
def fill_nonfinite(y):
    y = np.asarray(y, dtype=np.float32)
    good = np.isfinite(y)
    if good.all():
        return y
    if not good.any():
        raise ValueError("Trace contains no finite samples.")
    out = y.copy()
    index = np.arange(len(y))
    out[~good] = np.interp(index[~good], index[good], y[good])
    return out


def robust_mad(x):
    x = np.asarray(x, dtype=float)
    center = np.nanmedian(x)
    return float(1.4826 * np.nanmedian(np.abs(x - center)))


def quantile_baseline(y, fs, bin_s, percentile, smooth_s):
    y = np.asarray(y, dtype=np.float32)
    bin_samples = max(1, int(round(bin_s * fs)))
    n_full = y.size // bin_samples

    values = []
    centers = []
    if n_full:
        blocks = y[:n_full * bin_samples].reshape(n_full, bin_samples)
        values.extend(np.nanpercentile(blocks, percentile, axis=1))
        centers.extend((np.arange(n_full) + 0.5) * bin_samples - 0.5)
    if n_full * bin_samples < y.size:
        values.append(np.nanpercentile(y[n_full * bin_samples:], percentile))
        centers.append(0.5 * (n_full * bin_samples + y.size - 1))

    values = np.asarray(values, dtype=np.float32)
    centers = np.asarray(centers, dtype=float)

    if len(values) > 3:
        values = ndimage.gaussian_filter1d(
            values,
            sigma=max(0.5, smooth_s / bin_s),
            mode="nearest",
            truncate=3,
        )

    return np.interp(np.arange(y.size), centers, values).astype(np.float32)


def split_peak_indices(peaks, max_gap_samples):
    peaks = np.asarray(peaks, dtype=int)
    if peaks.size == 0:
        return []
    cuts = np.flatnonzero(np.diff(peaks) > max_gap_samples) + 1
    return list(np.split(np.arange(len(peaks), dtype=int), cuts))


def mask_to_intervals(mask):
    changes = np.diff(np.asarray(mask, dtype=np.int8), prepend=0, append=0)
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1)
    return [(int(start), int(stop)) for start, stop in zip(starts, stops)]


def merge_intervals(intervals, max_gap_samples):
    if not intervals:
        return []
    intervals = sorted(intervals)
    merged = [list(intervals[0])]
    for start, stop in intervals[1:]:
        if start - merged[-1][1] <= max_gap_samples:
            merged[-1][1] = max(merged[-1][1], stop)
        else:
            merged.append([start, stop])
    return [tuple(x) for x in merged]


def local_crossing_metrics(waveform, peak_index, fs):
    amplitude = waveform[peak_index]
    if not np.isfinite(amplitude) or amplitude <= 0:
        return np.nan, np.nan, np.nan

    def left_cross(frac):
        target = frac * amplitude
        for i in range(peak_index - 1, -1, -1):
            if waveform[i] <= target < waveform[i + 1]:
                return i + (
                    (target - waveform[i])
                    / (waveform[i + 1] - waveform[i])
                )
        return np.nan

    def right_cross(frac):
        target = frac * amplitude
        for i in range(peak_index, len(waveform) - 1):
            if waveform[i] >= target > waveform[i + 1]:
                return i + (
                    (waveform[i] - target)
                    / (waveform[i] - waveform[i + 1])
                )
        return np.nan

    left10 = left_cross(0.10)
    left50 = left_cross(0.50)
    left90 = left_cross(0.90)
    right50 = right_cross(0.50)
    right10 = right_cross(0.10)

    rise_ms = (
        (left90 - left10) / fs * 1000
        if np.isfinite(left10 + left90)
        else np.nan
    )
    width_ms = (
        (right50 - left50) / fs * 1000
        if np.isfinite(left50 + right50)
        else np.nan
    )
    decay_ms = (
        (right10 - peak_index) / fs * 1000
        if np.isfinite(right10)
        else np.nan
    )
    return rise_ms, width_ms, decay_ms


def find_depolarization_episodes(x, fs, params):
    sigma = max(
        0.1,
        params["burst_envelope_smooth_ms"] / 1000 * fs,
    )
    envelope = ndimage.gaussian_filter1d(
        x,
        sigma=sigma,
        mode="nearest",
        truncate=3,
    )
    threshold = (
        np.nanmean(envelope)
        + params["burst_envelope_height_sd"] * np.nanstd(envelope, ddof=1)
    )

    intervals = mask_to_intervals(envelope > threshold)
    intervals = merge_intervals(
        intervals,
        max_gap_samples=int(round(
            params["burst_envelope_merge_gap_ms"] / 1000 * fs
        )),
    )
    min_samples = int(round(
        params["burst_envelope_min_ms"] / 1000 * fs
    ))
    intervals = [
        (start, stop)
        for start, stop in intervals
        if stop - start >= min_samples
    ]
    return intervals, envelope, float(threshold)


def expand_burst_with_envelope(group, episodes, fs, params, n_samples):
    first = int(group[0])
    last = int(group[-1])

    overlaps = [
        (start, stop)
        for start, stop in episodes
        if stop > first and start <= last
    ]

    start = first
    stop = last + 1
    if overlaps:
        start = min(start, min(x[0] for x in overlaps))
        stop = max(stop, max(x[1] for x in overlaps))

    pre_limit = int(round(
        params["burst_expand_pre_max_ms"] / 1000 * fs
    ))
    post_limit = int(round(
        params["burst_expand_post_max_ms"] / 1000 * fs
    ))

    start = max(first - pre_limit, start, 0)
    stop = min(last + post_limit + 1, stop, n_samples)
    return int(start), int(stop)


In [ ]:
def detect_and_characterize_roi(
    y,
    fs,
    params,
    trial_slices=None,
):
    """Detect candidate spikes and classify singleton, doublet, and burst events."""
    y = fill_nonfinite(y)
    center = float(np.nanmean(y))
    scale = float(np.nanstd(y, ddof=1))

    distance = max(
        1,
        int(round(params["refractory_ms"] / 1000 * fs)),
    )
    min_width = max(
        1.0,
        params["min_width_ms"] / 1000 * fs,
    )
    prominence_wlen = max(
        3,
        int(round(params["prominence_window_ms"] / 1000 * fs)),
    )
    if prominence_wlen % 2 == 0:
        prominence_wlen += 1

    peaks, properties = signal.find_peaks(
        y,
        height=center + params["candidate_height_sd"] * scale,
        prominence=params["prominence_dff"],
        distance=distance,
        width=min_width,
        wlen=prominence_wlen,
    )

    edge_ms = params["exclude_trial_edge_ms"]
    if edge_ms > 0 and trial_slices:
        edge = int(round(edge_ms / 1000 * fs))
        keep = np.ones(len(peaks), dtype=bool)
        boundaries = np.asarray(
            [sl.stop for sl in trial_slices[:-1]],
            dtype=int,
        )
        for boundary in boundaries:
            keep &= np.abs(peaks - boundary) > edge
        peaks = peaks[keep]
        properties = {
            key: np.asarray(value)[keep]
            for key, value in properties.items()
        }

    high_confidence = (
        properties.get("peak_heights", np.array([]))
        >= center + params["waveform_height_sd"] * scale
    )

    group_gap = int(round(params["group_link_ms"] / 1000 * fs))
    group_indices = split_peak_indices(peaks, group_gap)

    baseline = quantile_baseline(
        y,
        fs,
        params["baseline_bin_s"],
        params["baseline_percentile"],
        params["baseline_smooth_s"],
    )
    x = y - baseline

    episodes, burst_envelope, burst_envelope_threshold = (
        find_depolarization_episodes(x, fs, params)
    )

    noise_residual = y - ndimage.gaussian_filter1d(
        y,
        sigma=max(0.1, 2.0 / 1000 * fs),
        mode="nearest",
        truncate=3,
    )
    noise = robust_mad(noise_residual)
    if not np.isfinite(noise) or noise <= 0:
        noise = float(np.nanstd(noise_residual, ddof=1))

    event_id = np.full(len(peaks), -1, dtype=int)
    spike_order = np.zeros(len(peaks), dtype=int)
    spike_class = np.full(len(peaks), "", dtype=object)

    event_rows = []
    burst_waveforms = []
    burst_relative_spikes_ms = []
    burst_event_ids = []

    burst_pre = int(round(
        params["burst_example_pre_ms"] / 1000 * fs
    ))
    burst_post = int(round(
        params["burst_example_post_ms"] / 1000 * fs
    ))

    for current_event_id, indices in enumerate(group_indices):
        group = peaks[indices]
        n_group = len(group)

        if n_group == 1:
            event_class = "singleton"
        elif n_group == 2:
            event_class = "doublet"
        else:
            event_class = "burst"

        event_id[indices] = current_event_id
        spike_order[indices] = np.arange(1, n_group + 1)
        spike_class[indices] = event_class

        first = int(group[0])
        last = int(group[-1])

        if event_class == "burst":
            onset, offset = expand_burst_with_envelope(
                group,
                episodes,
                fs,
                params,
                len(y),
            )
        else:
            onset, offset = first, last + 1

        event_rows.append({
            "event_id": current_event_id,
            "event_class": event_class,
            "event_onset_sample": onset,
            "first_spike_sample": first,
            "event_peak_sample": int(group[np.argmax(x[group])]),
            "last_spike_sample": last,
            "event_offset_sample": offset,
            "n_inferred_spikes": n_group,
            "spike_train_duration_ms": (last - first) / fs * 1000,
            "event_window_duration_ms": (offset - onset) / fs * 1000,
        })

        if (
            event_class == "burst"
            and first - burst_pre >= 0
            and first + burst_post < len(x)
        ):
            waveform = x[
                first - burst_pre:first + burst_post + 1
            ].copy()
            pre_segment = waveform[
                :max(3, int(round(0.6 * burst_pre)))
            ]
            waveform -= np.nanmedian(pre_segment)
            burst_waveforms.append(waveform)
            burst_relative_spikes_ms.append(
                (group - first) / fs * 1000
            )
            burst_event_ids.append(current_event_id)

    event_columns = [
        "event_id",
        "event_class",
        "event_onset_sample",
        "first_spike_sample",
        "event_peak_sample",
        "last_spike_sample",
        "event_offset_sample",
        "n_inferred_spikes",
        "spike_train_duration_ms",
        "event_window_duration_ms",
    ]
    events_df = pd.DataFrame(event_rows, columns=event_columns)

    plateau0 = int(round(
        params["plateau_window_ms"][0] / 1000 * fs
    ))
    plateau1 = int(round(
        params["plateau_window_ms"][1] / 1000 * fs
    ))
    plateau_index = np.full(len(peaks), np.nan)

    for i, peak in enumerate(peaks):
        amplitude = x[peak]
        if amplitude > 0 and peak + plateau1 <= len(x):
            plateau_index[i] = (
                np.nanmean(x[peak + plateau0:peak + plateau1])
                / amplitude
            )

    waveform_pre = int(round(
        params["waveform_pre_ms"] / 1000 * fs
    ))
    waveform_post = int(round(
        params["waveform_post_ms"] / 1000 * fs
    ))
    baseline_start = int(round(
        (params["waveform_baseline_ms"][0]
         + params["waveform_pre_ms"])
        / 1000 * fs
    ))
    baseline_stop = int(round(
        (params["waveform_baseline_ms"][1]
         + params["waveform_pre_ms"])
        / 1000 * fs
    ))

    amplitude_dff = np.full(len(peaks), np.nan)
    snr = np.full(len(peaks), np.nan)
    rise_ms = np.full(len(peaks), np.nan)
    width_ms = np.full(len(peaks), np.nan)
    decay_ms = np.full(len(peaks), np.nan)
    is_waveform_spike = np.zeros(len(peaks), dtype=bool)

    isolated_waveforms = []
    isolated_spike_indices = []

    for i, peak in enumerate(peaks):
        if spike_class[i] != "singleton" or not high_confidence[i]:
            continue
        if peak - waveform_pre < 0 or peak + waveform_post >= len(y):
            continue

        waveform = y[
            peak - waveform_pre:peak + waveform_post + 1
        ].copy()
        local_baseline = np.nanmedian(
            waveform[baseline_start:baseline_stop]
        )
        waveform -= local_baseline

        amplitude = waveform[waveform_pre]
        if not np.isfinite(amplitude) or amplitude <= 0:
            continue

        rise, width, decay = local_crossing_metrics(
            waveform,
            waveform_pre,
            fs,
        )
        amplitude_dff[i] = amplitude
        snr[i] = amplitude / noise if noise > 0 else np.nan
        rise_ms[i] = rise
        width_ms[i] = width
        decay_ms[i] = decay
        is_waveform_spike[i] = True
        isolated_waveforms.append(waveform)
        isolated_spike_indices.append(i)

    spike_rows = []
    for i, peak in enumerate(peaks):
        spike_rows.append({
            "spike_sample": int(peak),
            "event_id": int(event_id[i]),
            "spike_order": int(spike_order[i]),
            "spike_class": str(spike_class[i]),
            "is_high_confidence": bool(high_confidence[i]),
            "is_waveform_spike": bool(is_waveform_spike[i]),
            "is_burst_spike": bool(spike_class[i] == "burst"),
            "peak_height_dff": float(
                properties["peak_heights"][i]
            ),
            "prominence_dff": float(
                properties["prominences"][i]
            ),
            "find_peaks_width_ms": float(
                properties["widths"][i] / fs * 1000
            ),
            "amplitude_dff": float(amplitude_dff[i]),
            "snr": float(snr[i]),
            "plateau_index": float(plateau_index[i]),
            "rise10_90_ms": float(rise_ms[i]),
            "width50_ms": float(width_ms[i]),
            "decay90_10_ms": float(decay_ms[i]),
        })

    spike_columns = [
        "spike_sample",
        "event_id",
        "spike_order",
        "spike_class",
        "is_high_confidence",
        "is_waveform_spike",
        "is_burst_spike",
        "peak_height_dff",
        "prominence_dff",
        "find_peaks_width_ms",
        "amplitude_dff",
        "snr",
        "plateau_index",
        "rise10_90_ms",
        "width50_ms",
        "decay90_10_ms",
    ]
    spikes_df = pd.DataFrame(spike_rows, columns=spike_columns)

    waveform_length = waveform_pre + waveform_post + 1
    if isolated_waveforms:
        isolated_waveforms = np.asarray(
            isolated_waveforms,
            dtype=np.float32,
        )
    else:
        isolated_waveforms = np.empty(
            (0, waveform_length),
            dtype=np.float32,
        )

    burst_length = burst_pre + burst_post + 1
    if burst_waveforms:
        burst_waveforms = np.asarray(
            burst_waveforms,
            dtype=np.float32,
        )
    else:
        burst_waveforms = np.empty(
            (0, burst_length),
            dtype=np.float32,
        )

    return {
        "spikes": spikes_df,
        "events": events_df,
        "spike_samples": peaks,
        "isolated_waveforms": isolated_waveforms,
        "isolated_spike_indices": np.asarray(
            isolated_spike_indices,
            dtype=int,
        ),
        "burst_waveforms": burst_waveforms,
        "burst_relative_spikes_ms": burst_relative_spikes_ms,
        "burst_event_ids": np.asarray(burst_event_ids, dtype=int),
        "baseline": baseline,
        "baseline_subtracted": x,
        "burst_envelope": burst_envelope,
        "burst_envelope_threshold": burst_envelope_threshold,
        "candidate_height_threshold": (
            center + params["candidate_height_sd"] * scale
        ),
        "waveform_height_threshold": (
            center + params["waveform_height_sd"] * scale
        ),
        "trace_mean": center,
        "trace_sd": scale,
        "noise": noise,
    }


## 8. Run detection and build ROI-level metrics

`approx_spike_rate_hz` uses all candidate peaks. Waveform features use only high-confidence singleton spikes. Burst duration comes from the independently estimated slow envelope around ≥3-peak groups.


In [ ]:
analysis_results = {}
event_tables = []
spike_tables = []
metric_rows = []

waveform_pre = int(round(
    DETECTION["waveform_pre_ms"] / 1000 * FS
))
waveform_post = int(round(
    DETECTION["waveform_post_ms"] / 1000 * FS
))
waveform_time_ms = (
    np.arange(-waveform_pre, waveform_post + 1)
    / FS * 1000
)

burst_pre = int(round(
    DETECTION["burst_example_pre_ms"] / 1000 * FS
))
burst_post = int(round(
    DETECTION["burst_example_post_ms"] / 1000 * FS
))
burst_time_ms = (
    np.arange(-burst_pre, burst_post + 1)
    / FS * 1000
)

compressed_time = trace_info["compressed_time_sec"]
acquisition_time = trace_info["acquisition_time_sec"]

for dmd, x in traces.items():
    for roi, y in enumerate(x):
        result = detect_and_characterize_roi(
            y,
            FS,
            DETECTION,
            trial_slices=segment_slices,
        )

        events = result["events"].assign(
            subject_id=asset.subject_id,
            session_id=asset.session_id,
            dmd=dmd,
            roi=roi,
            depth_um=depths[dmd],
        )
        spikes = result["spikes"].assign(
            subject_id=asset.subject_id,
            session_id=asset.session_id,
            dmd=dmd,
            roi=roi,
            depth_um=depths[dmd],
        )

        if len(events):
            events["event_time_sec"] = compressed_time[
                events["event_onset_sample"].to_numpy(dtype=int)
            ]
            events["event_acquisition_time_sec"] = acquisition_time[
                events["event_onset_sample"].to_numpy(dtype=int)
            ]

        if len(spikes):
            spike_samples = spikes[
                "spike_sample"
            ].to_numpy(dtype=int)
            spikes["spike_time_sec"] = compressed_time[spike_samples]
            spikes["spike_acquisition_time_sec"] = (
                acquisition_time[spike_samples]
            )

        event_tables.append(events)
        spike_tables.append(spikes)

        isi = np.diff(result["spike_samples"]) / FS
        cv2 = (
            np.nanmean(
                2 * np.abs(np.diff(isi))
                / (isi[:-1] + isi[1:])
            )
            if len(isi) > 1
            else np.nan
        )

        bursts = events.loc[
            events["event_class"] == "burst"
        ] if len(events) else events
        singletons = events.loc[
            events["event_class"] == "singleton"
        ] if len(events) else events
        doublets = events.loc[
            events["event_class"] == "doublet"
        ] if len(events) else events
        waveform_spikes = spikes.loc[
            spikes["is_waveform_spike"]
        ] if len(spikes) else spikes

        burst_spike_fraction = (
            np.mean(spikes["spike_class"] == "burst")
            if len(spikes)
            else np.nan
        )
        doublet_spike_fraction = (
            np.mean(spikes["spike_class"] == "doublet")
            if len(spikes)
            else np.nan
        )
        singleton_spike_fraction = (
            np.mean(spikes["spike_class"] == "singleton")
            if len(spikes)
            else np.nan
        )

        burst_intervals = [
            (
                int(row.event_onset_sample),
                int(row.event_offset_sample),
            )
            for row in bursts.itertuples()
        ]
        burst_intervals = merge_intervals(
            burst_intervals,
            max_gap_samples=0,
        )
        burst_occupancy = (
            sum(stop - start for start, stop in burst_intervals)
            / traces[dmd].shape[1]
            if burst_intervals
            else 0.0
        )

        metric_rows.append({
            "subject_id": asset.subject_id,
            "session_id": asset.session_id,
            "dmd": dmd,
            "roi": roi,
            "label": f"DMD{dmd} ROI{roi}",
            "depth_um": depths[dmd],
            "duration_s": duration_s,

            "approx_spike_rate_hz": len(spikes) / duration_s,
            "event_rate_hz": len(events) / duration_s,
            "singleton_event_rate_hz": len(singletons) / duration_s,
            "doublet_event_rate_hz": len(doublets) / duration_s,
            "burst_event_rate_hz": len(bursts) / duration_s,

            "singleton_spike_fraction": singleton_spike_fraction,
            "doublet_spike_fraction": doublet_spike_fraction,
            "spike_burst_fraction": burst_spike_fraction,
            "burst_occupancy": burst_occupancy,

            "median_spikes_per_burst": (
                bursts["n_inferred_spikes"].median()
                if len(bursts)
                else np.nan
            ),
            "median_burst_duration_ms": (
                bursts["event_window_duration_ms"].median()
                if len(bursts)
                else np.nan
            ),
            "median_spike_train_duration_ms": (
                bursts["spike_train_duration_ms"].median()
                if len(bursts)
                else np.nan
            ),

            "isi_cv2": cv2,
            "short_isi_fraction_20ms": (
                np.mean(isi <= DETECTION["group_link_ms"] / 1000)
                if len(isi)
                else np.nan
            ),

            "median_isolated_amplitude_dff": (
                waveform_spikes["amplitude_dff"].median()
                if len(waveform_spikes)
                else np.nan
            ),
            "median_isolated_snr": (
                waveform_spikes["snr"].median()
                if len(waveform_spikes)
                else np.nan
            ),
            "median_isolated_rise10_90_ms": (
                waveform_spikes["rise10_90_ms"].median()
                if len(waveform_spikes)
                else np.nan
            ),
            "median_isolated_width50_ms": (
                waveform_spikes["width50_ms"].median()
                if len(waveform_spikes)
                else np.nan
            ),
            "median_isolated_decay90_10_ms": (
                waveform_spikes["decay90_10_ms"].median()
                if len(waveform_spikes)
                else np.nan
            ),

            "median_plateau_index": (
                spikes["plateau_index"].median()
                if len(spikes)
                else np.nan
            ),

            "n_spikes": len(spikes),
            "n_events": len(events),
            "n_singletons": len(singletons),
            "n_doublets": len(doublets),
            "n_bursts": len(bursts),
            "n_waveform_spikes": len(waveform_spikes),
        })

        result["events"] = events
        result["spikes"] = spikes
        result["spike_times_sec"] = (
            spikes["spike_time_sec"].to_numpy()
            if len(spikes)
            else np.array([])
        )
        result["isolated_spike_times_sec"] = (
            waveform_spikes["spike_time_sec"].to_numpy()
            if len(waveform_spikes)
            else np.array([])
        )
        result["burst_onset_times_sec"] = (
            bursts["event_time_sec"].to_numpy()
            if len(bursts)
            else np.array([])
        )

        analysis_results[(dmd, roi)] = result

event_df = pd.concat(event_tables, ignore_index=True)
spike_df = pd.concat(spike_tables, ignore_index=True)

roi_metrics = roi_manifest.merge(
    pd.DataFrame(metric_rows),
    on=[
        "subject_id",
        "session_id",
        "dmd",
        "roi",
        "depth_um",
    ],
    how="left",
)

display(roi_metrics.round(3))


## 9. Detection QC

Triangles show all candidate spikes. Open circles identify the high-confidence singleton spikes used for waveform analysis. Pale bands show independently estimated burst windows.


In [ ]:
# Editable diagnostic: compare three height thresholds on one ROI.
QC_DMD = 2
QC_ROI = min(3, traces[2].shape[0] - 1)
QC_WINDOW_SEC = ZOOM_SEC
HEIGHT_SD_LEVELS = (1.0, 1.5, 2.0)

q0, q1 = QC_WINDOW_SEC
j0 = max(0, int(round(q0 * FS)))
j1 = min(traces[QC_DMD].shape[1], int(round(q1 * FS)))

y = traces[QC_DMD][QC_ROI]
window = y[j0:j1]
time = np.arange(j0, j1) / FS
vertical_range = (
    np.nanpercentile(window, 99)
    - np.nanpercentile(window, 1)
)
offset_step = 1.30 * vertical_range

fig, ax = plt.subplots(figsize=(12.5, 6.2))

sensitivity_colors = [TEAL, PEACH, BLUE]
for level, (height_sd, color) in enumerate(
    zip(HEIGHT_SD_LEVELS, sensitivity_colors)
):
    wlen = max(
        3,
        int(round(
            DETECTION["prominence_window_ms"] / 1000 * FS
        )),
    )
    if wlen % 2 == 0:
        wlen += 1

    peaks, _ = signal.find_peaks(
        y,
        height=np.nanmean(y) + height_sd * np.nanstd(y, ddof=1),
        prominence=DETECTION["prominence_dff"],
        distance=max(
            1,
            int(round(
                DETECTION["refractory_ms"] / 1000 * FS
            )),
        ),
        width=max(
            1.0,
            DETECTION["min_width_ms"] / 1000 * FS,
        ),
        wlen=wlen,
    )
    selected = peaks[(peaks >= j0) & (peaks < j1)]
    offset = level * offset_step

    ax.plot(
        time,
        window + offset,
        color=color,
        lw=0.75,
        zorder=ZORDER["trace"],
    )
    ax.scatter(
        selected / FS,
        y[selected] + offset,
        marker="v",
        s=22,
        facecolor=CHARCOAL,
        edgecolor="white",
        linewidth=0.35,
        zorder=ZORDER["marker"],
    )
    ax.text(
        q0 - 0.015 * (q1 - q0),
        offset,
        f"{height_sd:g} SD · {len(selected)} peaks",
        ha="right",
        va="center",
        color=CHARCOAL,
        fontsize=10,
    )

ax.set_xlim(q0, q1)
ax.set_yticks([])
ax.set_xlabel("Time (s)")
ax.set_title(
    f"DMD{QC_DMD} ROI{QC_ROI}: candidate-height sensitivity"
)
fig.tight_layout()
save_panel(fig, "04a_detection_threshold_sensitivity")
plt.show()


In [ ]:
q0, q1 = QC_WINDOW_SEC
j0 = max(0, int(round(q0 * FS)))
j1 = min(traces[1].shape[1], int(round(q1 * FS)))

fig, axes = plt.subplots(
    len(traces),
    1,
    figsize=(
        13,
        2.5 + 0.90 * sum(x.shape[0] for x in traces.values()),
    ),
    sharex=True,
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    window = traces[dmd][:, j0:j1]
    scale = np.nanmedian(
        np.nanpercentile(window, 95, axis=1)
        - np.nanpercentile(window, 5, axis=1)
    )
    offsets = np.arange(window.shape[0]) * scale * 1.55
    time = np.arange(j0, j1) / FS

    for roi, row in enumerate(window):
        centered = row - np.nanmedian(row)
        offset = offsets[roi]
        result = analysis_results[(dmd, roi)]

        bursts = result["events"].loc[
            result["events"]["event_class"] == "burst"
        ]
        bursts = bursts.loc[
            (bursts["event_offset_sample"] >= j0)
            & (bursts["event_onset_sample"] < j1)
        ]
        for event in bursts.itertuples():
            left = max(j0, event.event_onset_sample) / FS
            right = min(j1, event.event_offset_sample) / FS
            ax.fill_between(
                [left, right],
                [offset - 0.48 * scale] * 2,
                [offset + 0.48 * scale] * 2,
                color=CREAM,
                alpha=0.55,
                linewidth=0,
                zorder=ZORDER["span"],
            )

        ax.plot(
            time,
            centered + offset,
            lw=0.65,
            color=DMD_COLORS[dmd],
            zorder=ZORDER["trace"],
        )

        spikes = result["spikes"]
        selected = spikes.loc[
            (spikes["spike_sample"] >= j0)
            & (spikes["spike_sample"] < j1)
        ]
        samples = selected["spike_sample"].to_numpy(dtype=int)
        ax.scatter(
            samples / FS,
            traces[dmd][roi, samples]
            - np.nanmedian(row)
            + offset,
            marker="v",
            s=18,
            facecolor=CHARCOAL,
            edgecolor="white",
            linewidth=0.3,
            zorder=ZORDER["marker"],
        )

        waveform_calls = selected.loc[
            selected["is_waveform_spike"]
        ]
        waveform_samples = waveform_calls[
            "spike_sample"
        ].to_numpy(dtype=int)
        ax.scatter(
            waveform_samples / FS,
            traces[dmd][roi, waveform_samples]
            - np.nanmedian(row)
            + offset,
            marker="o",
            s=42,
            facecolor="white",
            edgecolor=DMD_DARK[dmd],
            linewidth=1.3,
            zorder=ZORDER["annotation"],
        )

        ax.text(
            q0 - 0.012 * (q1 - q0),
            offset,
            f"ROI {roi}",
            ha="right",
            va="center",
            color=CHARCOAL,
        )

    ax.set_yticks([])
    ax.set_ylabel(
        f"DMD{dmd}\n{depths[dmd]:.0f} µm"
    )
    ax.set_title(f"DMD{dmd}: spike and burst detection")

axes[-1].set_xlabel("Time (s)")

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="v",
        linestyle="none",
        markerfacecolor=CHARCOAL,
        markeredgecolor="white",
        markersize=7,
        label="Candidate spike",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="white",
        markeredgecolor=NAVY,
        markeredgewidth=1.3,
        markersize=7,
        label="Waveform-quality singleton",
    ),
    Patch(
        facecolor=CREAM,
        edgecolor="none",
        alpha=0.75,
        label="Burst window",
    ),
]
axes[0].legend(
    handles=legend_handles,
    loc="upper right",
    ncol=3,
)

fig.tight_layout()
save_panel(fig, "04b_detection_qc")
plt.show()


## 10. High-confidence isolated optical spike waveforms

Only ≥2-SD singleton peaks are included. Each waveform is locally baseline-subtracted and normalized to its own peak. The dark line and pastel band show the median and 10th–90th percentiles.


In [ ]:
n_rois_max = max(x.shape[0] for x in traces.values())

fig, axes = plt.subplots(
    len(traces),
    n_rois_max,
    figsize=(3.2 * n_rois_max, 2.8 * len(traces)),
    sharex=True,
)
axes = np.atleast_2d(axes)

for row_index, dmd in enumerate(traces):
    for roi in range(n_rois_max):
        ax = axes[row_index, roi]

        if roi >= traces[dmd].shape[0]:
            ax.axis("off")
            continue

        waveforms = analysis_results[
            (dmd, roi)
        ]["isolated_waveforms"]

        if len(waveforms):
            amplitudes = waveforms[:, waveform_pre]
            keep = np.isfinite(amplitudes) & (amplitudes > 0)
            normalized = waveforms[keep] / amplitudes[keep, None]

            if len(normalized) > 500:
                selected = np.linspace(
                    0,
                    len(normalized) - 1,
                    500,
                    dtype=int,
                )
                normalized = normalized[selected]

            lower, median, upper = np.nanpercentile(
                normalized,
                [10, 50, 90],
                axis=0,
            )

            ax.fill_between(
                waveform_time_ms,
                lower,
                upper,
                color=DMD_FILLS[dmd],
                alpha=0.65,
                linewidth=0,
                zorder=ZORDER["span"],
            )
            ax.plot(
                waveform_time_ms,
                median,
                color=DMD_DARK[dmd],
                lw=2.4,
                zorder=ZORDER["summary"],
            )

        ax.axvline(
            0,
            color=LIGHT_GRAY,
            lw=0.9,
            zorder=ZORDER["grid"],
        )
        ax.axhline(
            0,
            color=LIGHT_GRAY,
            lw=0.9,
            zorder=ZORDER["grid"],
        )
        ax.text(
            0.97,
            0.94,
            f"n = {len(waveforms):,}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9,
            color=CHARCOAL,
            zorder=ZORDER["annotation"],
        )
        ax.set_xlim(
            -DETECTION["waveform_pre_ms"],
            DETECTION["waveform_post_ms"],
        )
        ax.set_title(f"DMD{dmd} ROI{roi}")

        if roi == 0:
            ax.set_ylabel("Normalized dF/F")
        if row_index == len(traces) - 1:
            ax.set_xlabel("Time from spike (ms)")

fig.suptitle(
    "High-confidence isolated ASAP8 optical spikes",
    color=NAVY,
    y=1.01,
)
fig.tight_layout()
save_panel(fig, "05_isolated_spike_waveforms")
plt.show()


## 11. Representative burst structure

Each panel shows representative ≥3-spike events aligned to the first inferred spike. Candidate spike times remain visible as dark triangles; the traces are not averaged.


In [ ]:
N_BURSTS_TO_SHOW = 5

fig, axes = plt.subplots(
    len(traces),
    n_rois_max,
    figsize=(3.25 * n_rois_max, 3.05 * len(traces)),
    sharex=True,
)
axes = np.atleast_2d(axes)

for row_index, dmd in enumerate(traces):
    for roi in range(n_rois_max):
        ax = axes[row_index, roi]

        if roi >= traces[dmd].shape[0]:
            ax.axis("off")
            continue

        result = analysis_results[(dmd, roi)]
        waveforms = result["burst_waveforms"]
        relative_spikes = result["burst_relative_spikes_ms"]
        event_ids = result["burst_event_ids"]
        events = result["events"].set_index("event_id")

        if len(waveforms):
            ordering_values = np.asarray([
                events.loc[event_id, "n_inferred_spikes"]
                + events.loc[event_id, "event_window_duration_ms"] / 100
                for event_id in event_ids
            ])
            order = np.argsort(ordering_values)

            if len(order) <= N_BURSTS_TO_SHOW:
                selected = order
            else:
                selected = order[
                    np.linspace(
                        0,
                        len(order) - 1,
                        N_BURSTS_TO_SHOW,
                        dtype=int,
                    )
                ]

            selected_waveforms = waveforms[selected]
            robust_range = np.nanmedian(
                np.nanpercentile(selected_waveforms, 98, axis=1)
                - np.nanpercentile(selected_waveforms, 2, axis=1)
            )
            spacing = max(robust_range * 1.25, 1e-6)

            for level, index in enumerate(selected):
                waveform = waveforms[index]
                offset = level * spacing
                event_id = int(event_ids[index])
                event = events.loc[event_id]

                ax.plot(
                    burst_time_ms,
                    waveform + offset,
                    color=DMD_COLORS[dmd],
                    lw=1.15,
                    zorder=ZORDER["trace"],
                )

                spike_times = np.asarray(
                    relative_spikes[index],
                    dtype=float,
                )
                spike_values = np.interp(
                    spike_times,
                    burst_time_ms,
                    waveform,
                )
                ax.scatter(
                    spike_times,
                    spike_values + offset,
                    marker="v",
                    s=20,
                    facecolor=CHARCOAL,
                    edgecolor="white",
                    linewidth=0.3,
                    zorder=ZORDER["marker"],
                )
                ax.text(
                    DETECTION["burst_example_post_ms"] - 1,
                    offset,
                    (
                        f"{int(event['n_inferred_spikes'])} spikes · "
                        f"{event['event_window_duration_ms']:.0f} ms"
                    ),
                    ha="right",
                    va="bottom",
                    fontsize=8,
                    color=CHARCOAL,
                    zorder=ZORDER["annotation"],
                )

        ax.axvline(
            0,
            color=LIGHT_GRAY,
            lw=0.9,
            zorder=ZORDER["grid"],
        )
        ax.set_xlim(
            -DETECTION["burst_example_pre_ms"],
            DETECTION["burst_example_post_ms"],
        )
        ax.set_yticks([])
        ax.set_title(f"DMD{dmd} ROI{roi}")

        if roi == 0:
            ax.set_ylabel("Representative bursts")
        if row_index == len(traces) - 1:
            ax.set_xlabel("Time from first spike (ms)")

fig.suptitle(
    "ASAP8 burst structure",
    color=NAVY,
    y=1.01,
)
fig.tight_layout()
save_panel(fig, "06_representative_bursts")
plt.show()


## 12. Figure-ready ROI phenotype panels

The core comparison separates approximate firing rate, clean isolated-spike shape, burst composition, burst duration, and the original simple plateau index.


In [ ]:
plot_metrics = [
    ("approx_spike_rate_hz", "Approximate spike rate (Hz)"),
    (
        "median_isolated_width50_ms",
        "Isolated spike width at half-height (ms)",
    ),
    (
        "spike_burst_fraction",
        "Fraction of detected spikes in bursts",
    ),
    (
        "median_burst_duration_ms",
        "Median burst-window duration (ms)",
    ),
    ("median_plateau_index", "Median plateau index"),
    ("median_isolated_snr", "Isolated spike SNR"),
]

fig, axes = plt.subplots(2, 3, figsize=(12.8, 7.2))
rng = np.random.default_rng(4)

for ax, (metric, ylabel) in zip(axes.flat, plot_metrics):
    for position, dmd in enumerate(sorted(traces), start=1):
        values = roi_metrics.loc[
            roi_metrics["dmd"] == dmd,
            metric,
        ].dropna().to_numpy()

        if len(values):
            median = np.nanmedian(values)
            ax.plot(
                [position - 0.20, position + 0.20],
                [median, median],
                color=DMD_DARK[dmd],
                lw=2.8,
                solid_capstyle="round",
                zorder=ZORDER["summary"],
            )

        jitter = rng.normal(0, 0.045, size=len(values))
        ax.scatter(
            position + jitter,
            values,
            s=64,
            color=DMD_FILLS[dmd],
            edgecolor=DMD_DARK[dmd],
            linewidth=1.1,
            zorder=ZORDER["marker"],
        )

    ax.set_xticks([1, 2])
    ax.set_xticklabels([
        f"DMD1\n{depths[1]:.0f} µm",
        f"DMD2\n{depths[2]:.0f} µm",
    ])
    ax.set_ylabel(ylabel)
    ax.grid(
        axis="y",
        color=LIGHT_GRAY,
        linewidth=0.7,
        alpha=0.55,
        zorder=ZORDER["grid"],
    )
    ax.margins(x=0.28)

fig.suptitle(
    "ASAP8 somatic voltage phenotype within one session",
    color=NAVY,
    y=1.01,
)
fig.tight_layout()
save_panel(fig, "07_roi_phenotype_summary")
plt.show()


In [ ]:
import seaborn as sns

fig,ax=plt.subplots()

metric = 'median_isolated_rise10_90_ms'

sns.swarmplot(data = roi_metrics, x = 'dmd', y = metric)

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 5.7))

max_fraction = max(
    float(roi_metrics["spike_burst_fraction"].max()),
    1e-9,
)

for dmd, subset in roi_metrics.groupby("dmd"):
    sizes = (
        70
        + 330
        * subset["spike_burst_fraction"].fillna(0)
        / max_fraction
    )
    ax.scatter(
        subset["approx_spike_rate_hz"],
        subset["median_plateau_index"],
        s=sizes,
        color=DMD_FILLS[dmd],
        edgecolor=DMD_DARK[dmd],
        linewidth=1.3,
        alpha=0.95,
        label=f"DMD{dmd} · {depths[dmd]:.0f} µm",
        zorder=ZORDER["marker"],
    )

    for row in subset.itertuples():
        ax.annotate(
            f"R{int(row.roi)}",
            (
                row.approx_spike_rate_hz,
                row.median_plateau_index,
            ),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
            color=CHARCOAL,
            zorder=ZORDER["annotation"],
        )

ax.grid(
    color=LIGHT_GRAY,
    linewidth=0.7,
    alpha=0.55,
    zorder=ZORDER["grid"],
)
ax.set_xlabel("Approximate detected-spike rate (Hz)")
ax.set_ylabel("Median plateau index")
ax.set_title("Firing, bursting, and sustained depolarization")
ax.legend(
    title="Point size: burst-spike fraction",
    loc="best",
)
fig.tight_layout()
save_panel(fig, "08_spike_rate_vs_plateau")
plt.show()


In [ ]:
display(
    roi_metrics[[
        "label",
        "depth_um",
        "approx_spike_rate_hz",
        "singleton_spike_fraction",
        "doublet_spike_fraction",
        "spike_burst_fraction",
        "burst_event_rate_hz",
        "median_spikes_per_burst",
        "median_burst_duration_ms",
        "median_plateau_index",
        "median_isolated_width50_ms",
        "n_waveform_spikes",
    ]].round(3)
)


# Part II — Correlated activity and synchrony

Approximate spike timing and burst timing are analyzed separately:

- candidate-spike STTC at ±5 ms;
- high-confidence singleton STTC at ±5 ms;
- burst-onset STTC at ±20 ms;
- spike-count correlations at 20 and 250 ms;
- overlap of burst-state occupancy.


## 13. Synchrony helpers


In [ ]:
def fraction_spikes_near(a, b, dt):
    if len(a) == 0 or len(b) == 0:
        return np.nan

    b = np.sort(np.asarray(b))
    hits = 0

    for time in np.asarray(a):
        index = np.searchsorted(b, time)
        near = (
            (
                index < len(b)
                and abs(b[index] - time) <= dt
            )
            or (
                index > 0
                and abs(b[index - 1] - time) <= dt
            )
        )
        hits += near

    return hits / len(a)


def fraction_time_tiled(times, dt, t_start, t_stop):
    if len(times) == 0:
        return 0.0

    intervals = np.c_[
        np.maximum(np.asarray(times) - dt, t_start),
        np.minimum(np.asarray(times) + dt, t_stop),
    ]
    intervals = intervals[np.argsort(intervals[:, 0])]

    covered = 0.0
    start, stop = intervals[0]

    for new_start, new_stop in intervals[1:]:
        if new_start <= stop:
            stop = max(stop, new_stop)
        else:
            covered += stop - start
            start, stop = new_start, new_stop

    covered += stop - start
    return covered / (t_stop - t_start)


def sttc(a, b, dt, t_start, t_stop):
    if len(a) == 0 or len(b) == 0:
        return np.nan

    pa = fraction_spikes_near(a, b, dt)
    pb = fraction_spikes_near(b, a, dt)
    ta = fraction_time_tiled(a, dt, t_start, t_stop)
    tb = fraction_time_tiled(b, dt, t_start, t_stop)

    term_a = (pa - tb) / (1 - pa * tb + 1e-12)
    term_b = (pb - ta) / (1 - pb * ta + 1e-12)
    return 0.5 * (term_a + term_b)


def binned_rate(times, duration, bin_s):
    edges = np.arange(0, duration + bin_s, bin_s)
    counts, _ = np.histogram(times, bins=edges)
    return counts / bin_s


def safe_correlation(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if (
        len(a) < 2
        or len(b) < 2
        or np.nanstd(a) == 0
        or np.nanstd(b) == 0
    ):
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def burst_state(events, duration, bin_s=0.010):
    n_bins = int(np.ceil(duration / bin_s))
    state = np.zeros(n_bins, dtype=bool)

    bursts = events.loc[
        events["event_class"] == "burst"
    ]
    for event in bursts.itertuples():
        start = max(
            0,
            int(np.floor(
                event.event_onset_sample / FS / bin_s
            )),
        )
        stop = min(
            n_bins,
            int(np.ceil(
                event.event_offset_sample / FS / bin_s
            )),
        )
        state[start:stop] = True

    return state


def jaccard_binary(a, b):
    union = np.logical_or(a, b).sum()
    if union == 0:
        return np.nan
    return np.logical_and(a, b).sum() / union


def cross_correlogram(
    a,
    b,
    window_s=0.100,
    bin_s=0.002,
):
    differences = []
    b = np.sort(np.asarray(b))

    for time in np.asarray(a):
        low = np.searchsorted(b, time - window_s)
        high = np.searchsorted(
            b,
            time + window_s,
            side="right",
        )
        differences.extend(b[low:high] - time)

    edges = np.arange(
        -window_s,
        window_s + bin_s,
        bin_s,
    )
    counts, _ = np.histogram(differences, bins=edges)
    centers = edges[:-1] + bin_s / 2
    conditional_rate = counts / (max(len(a), 1) * bin_s)
    return centers, conditional_rate


## 14. Pairwise synchrony table


In [ ]:
units = []

for dmd, x in traces.items():
    for roi in range(x.shape[0]):
        result = analysis_results[(dmd, roi)]
        units.append({
            "dmd": dmd,
            "roi": roi,
            "label": f"D{dmd} R{roi}",
            "spike_times": result["spike_times_sec"],
            "isolated_spike_times": (
                result["isolated_spike_times_sec"]
            ),
            "burst_onset_times": (
                result["burst_onset_times_sec"]
            ),
            "burst_state": burst_state(
                result["events"],
                duration_s,
                bin_s=0.010,
            ),
        })

pair_rows = []

for index_a, unit_a in enumerate(units):
    for index_b in range(index_a + 1, len(units)):
        unit_b = units[index_b]

        pair_class = (
            f"within DMD{unit_a['dmd']}"
            if unit_a["dmd"] == unit_b["dmd"]
            else "cross-DMD"
        )

        rate20_a = binned_rate(
            unit_a["spike_times"],
            duration_s,
            0.020,
        )
        rate20_b = binned_rate(
            unit_b["spike_times"],
            duration_s,
            0.020,
        )
        rate250_a = binned_rate(
            unit_a["spike_times"],
            duration_s,
            0.250,
        )
        rate250_b = binned_rate(
            unit_b["spike_times"],
            duration_s,
            0.250,
        )

        pair_rows.append({
            "session_id": asset.session_id,
            "unit_a": unit_a["label"],
            "unit_b": unit_b["label"],
            "dmd_a": unit_a["dmd"],
            "roi_a": unit_a["roi"],
            "dmd_b": unit_b["dmd"],
            "roi_b": unit_b["roi"],
            "pair_class": pair_class,

            "spike_sttc_5ms": sttc(
                unit_a["spike_times"],
                unit_b["spike_times"],
                0.005,
                0,
                duration_s,
            ),
            "isolated_spike_sttc_5ms": sttc(
                unit_a["isolated_spike_times"],
                unit_b["isolated_spike_times"],
                0.005,
                0,
                duration_s,
            ),
            "burst_onset_sttc_20ms": sttc(
                unit_a["burst_onset_times"],
                unit_b["burst_onset_times"],
                0.020,
                0,
                duration_s,
            ),
            "spike_count_corr_20ms": safe_correlation(
                rate20_a,
                rate20_b,
            ),
            "spike_count_corr_250ms": safe_correlation(
                rate250_a,
                rate250_b,
            ),
            "burst_state_jaccard": jaccard_binary(
                unit_a["burst_state"],
                unit_b["burst_state"],
            ),
        })

pair_df = pd.DataFrame(pair_rows)

display(
    pair_df.sort_values(
        "spike_sttc_5ms",
        ascending=False,
    ).round(3)
)


## 15. Spike raster and synchrony matrices


In [ ]:
RASTER_WINDOW_SEC = ZOOM_SEC
r0, r1 = RASTER_WINDOW_SEC

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15.0, 5.0),
    gridspec_kw={"width_ratios": [1.45, 1.0, 1.0]},
)

for row, unit in enumerate(units):
    spike_times = unit["spike_times"]
    spike_times = spike_times[
        (spike_times >= r0) & (spike_times <= r1)
    ]
    axes[0].vlines(
        spike_times,
        row - 0.34,
        row + 0.34,
        color=DMD_DARK[unit["dmd"]],
        lw=0.95,
        zorder=ZORDER["trace"],
    )

    burst_times = unit["burst_onset_times"]
    burst_times = burst_times[
        (burst_times >= r0) & (burst_times <= r1)
    ]
    axes[0].scatter(
        burst_times,
        np.full(len(burst_times), row),
        marker="o",
        s=26,
        facecolor=DMD_FILLS[unit["dmd"]],
        edgecolor=DMD_DARK[unit["dmd"]],
        linewidth=0.8,
        zorder=ZORDER["marker"],
    )

axes[0].set_yticks(np.arange(len(units)))
axes[0].set_yticklabels([unit["label"] for unit in units])
axes[0].set_xlim(r0, r1)
axes[0].set_xlabel("Time (s)")
axes[0].set_title("Candidate spikes and burst onsets")
axes[0].invert_yaxis()

labels = [unit["label"] for unit in units]

sttc_matrix = np.eye(len(units))
burst_matrix = np.eye(len(units))

for row in pair_df.itertuples():
    i = labels.index(row.unit_a)
    j = labels.index(row.unit_b)
    sttc_matrix[i, j] = sttc_matrix[j, i] = (
        row.spike_sttc_5ms
    )
    burst_matrix[i, j] = burst_matrix[j, i] = (
        row.burst_state_jaccard
    )

off_diagonal = sttc_matrix[np.eye(len(units)) == 0]
sttc_vmax = max(0.25, np.nanmax(off_diagonal))

image_sttc = axes[1].imshow(
    sttc_matrix,
    cmap=STTC_CMAP,
    vmin=0,
    vmax=sttc_vmax,
    zorder=ZORDER["trace"],
)
axes[1].set_title("Candidate-spike STTC · ±5 ms")

image_burst = axes[2].imshow(
    burst_matrix,
    cmap=STTC_CMAP,
    vmin=0,
    vmax=max(0.25, np.nanmax(
        burst_matrix[np.eye(len(units)) == 0]
    )),
    zorder=ZORDER["trace"],
)
axes[2].set_title("Burst-state overlap")

for ax in axes[1:]:
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

fig.colorbar(
    image_sttc,
    ax=axes[1],
    label="STTC",
    fraction=0.046,
    pad=0.04,
)
fig.colorbar(
    image_burst,
    ax=axes[2],
    label="Jaccard overlap",
    fraction=0.046,
    pad=0.04,
)

fig.tight_layout()
save_panel(fig, "09_spike_raster_and_synchrony")
plt.show()


## 16. Pairwise synchrony across timescales


In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(12.8, 4.4),
    sharex=True,
)

metrics = [
    ("spike_sttc_5ms", "Candidate-spike STTC · ±5 ms"),
    ("burst_onset_sttc_20ms", "Burst-onset STTC · ±20 ms"),
    (
        "spike_count_corr_250ms",
        "Spike-count correlation · 250 ms",
    ),
]

class_order = [
    "within DMD1",
    "within DMD2",
    "cross-DMD",
]
class_colors = {
    "within DMD1": BLUE_FILL,
    "within DMD2": PEACH_FILL,
    "cross-DMD": "#E4E4E1",
}
class_edges = {
    "within DMD1": DARK_BLUE,
    "within DMD2": DARK_PEACH,
    "cross-DMD": GRAY,
}
rng = np.random.default_rng(7)

for ax, (metric, title) in zip(axes, metrics):
    for position, pair_class in enumerate(
        class_order,
        start=1,
    ):
        values = pair_df.loc[
            pair_df["pair_class"] == pair_class,
            metric,
        ].dropna().to_numpy()

        if len(values):
            median = np.nanmedian(values)
            ax.plot(
                [position - 0.19, position + 0.19],
                [median, median],
                color=class_edges[pair_class],
                lw=2.8,
                solid_capstyle="round",
                zorder=ZORDER["summary"],
            )

        jitter = rng.normal(0, 0.05, size=len(values))
        ax.scatter(
            position + jitter,
            values,
            s=50,
            color=class_colors[pair_class],
            edgecolor=class_edges[pair_class],
            linewidth=1.0,
            zorder=ZORDER["marker"],
        )

    ax.axhline(
        0,
        color=LIGHT_GRAY,
        lw=0.9,
        zorder=ZORDER["grid"],
    )
    ax.grid(
        axis="y",
        color=LIGHT_GRAY,
        linewidth=0.7,
        alpha=0.55,
        zorder=ZORDER["grid"],
    )
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["DMD1", "DMD2", "cross"])
    ax.set_title(title)

fig.suptitle(
    "Pairwise synchrony across timescales",
    color=NAVY,
    y=1.02,
)
fig.tight_layout()
save_panel(fig, "10_pairwise_synchrony_timescales")
plt.show()


## 17. Cross-correlograms for the strongest within-DMD pair


In [ ]:
fig, axes = plt.subplots(
    1,
    len(traces),
    figsize=(5.8 * len(traces), 4.2),
    sharey=True,
)
axes = np.atleast_1d(axes)

for ax, dmd in zip(axes, traces):
    subset = pair_df.loc[
        pair_df["pair_class"] == f"within DMD{dmd}"
    ]
    best = subset.loc[subset["spike_sttc_5ms"].idxmax()]

    unit_a = next(
        unit
        for unit in units
        if unit["label"] == best["unit_a"]
    )
    unit_b = next(
        unit
        for unit in units
        if unit["label"] == best["unit_b"]
    )

    lag, conditional_rate = cross_correlogram(
        unit_a["spike_times"],
        unit_b["spike_times"],
        window_s=0.050,
        bin_s=0.001,
    )
    baseline_rate = (
        len(unit_b["spike_times"]) / duration_s
    )

    ax.fill_between(
        lag * 1000,
        conditional_rate - baseline_rate,
        0,
        color=DMD_FILLS[dmd],
        alpha=0.65,
        linewidth=0,
        zorder=ZORDER["span"],
    )
    ax.plot(
        lag * 1000,
        conditional_rate - baseline_rate,
        color=DMD_DARK[dmd],
        lw=2.0,
        zorder=ZORDER["summary"],
    )
    ax.axvline(
        0,
        color=LIGHT_GRAY,
        lw=0.9,
        zorder=ZORDER["grid"],
    )
    ax.axhline(
        0,
        color=LIGHT_GRAY,
        lw=0.9,
        zorder=ZORDER["grid"],
    )

    ax.set_title(
        f"DMD{dmd}: {best['unit_a']} ↔ {best['unit_b']}\n"
        f"STTC = {best['spike_sttc_5ms']:.2f}"
    )
    ax.set_xlabel("Lag (ms)")
    ax.set_ylabel("Excess conditional spike rate (Hz)")

fig.tight_layout()
save_panel(fig, "11_within_dmd_cross_correlograms")
plt.show()


## 18. Save canonical outputs


In [ ]:
if SAVE_TABLES:
    roi_manifest.to_csv(
        TABLE_DIR / "roi_session_manifest.csv",
        index=False,
    )
    roi_metrics.to_csv(
        TABLE_DIR / "asap8_roi_ephys_metrics.csv",
        index=False,
    )
    event_df.to_csv(
        TABLE_DIR / "asap8_detected_events.csv",
        index=False,
    )
    spike_df.to_csv(
        TABLE_DIR / "asap8_detected_spikes.csv",
        index=False,
    )
    pair_df.to_csv(
        TABLE_DIR / "asap8_pairwise_synchrony.csv",
        index=False,
    )

    isolated_waveforms = {
        f"DMD{dmd}_ROI{roi}": result["isolated_waveforms"]
        for (dmd, roi), result in analysis_results.items()
    }
    burst_waveforms = {
        f"DMD{dmd}_ROI{roi}": result["burst_waveforms"]
        for (dmd, roi), result in analysis_results.items()
    }

    np.savez_compressed(
        TABLE_DIR / "asap8_isolated_spike_waveforms.npz",
        **isolated_waveforms,
    )
    np.savez_compressed(
        TABLE_DIR / "asap8_burst_waveforms.npz",
        **burst_waveforms,
    )

print("Saved outputs to:", OUT_DIR)


# Part III — Longitudinal ROI registration

This section is intentionally reserved for the next analysis pass. The current notebook already writes the required single-session keys and geometry:

- `subject_id`
- `session_id`
- `dmd`
- `roi`
- ROI centroid and area
- cortical depth
- electrophysiological metrics

The registration extension should:

1. collect reference images and masks for all registry sessions from one subject;
2. register each DMD to a reference session using image content and acquisition geometry;
3. transform masks into the reference plane;
4. propose ROI matches using mask overlap, centroid distance, and image correlation;
5. expose an editable match table before assigning persistent `cell_id` values;
6. merge `cell_id` back into `asap8_roi_ephys_metrics.csv` using `session_id`, `dmd`, and `roi`.

## 19. Sessions queued for registration

In [ ]:
registration_sessions = session_df.copy()
registration_sessions["selected_for_registration"] = True
registration_sessions["reference_session"] = (
    registration_sessions["session_id"].astype(str) == str(asset.session_id)
)

display(registration_sessions[show_cols + ["reference_session"]])

if SAVE_TABLES:
    registration_sessions.to_csv(
        TABLE_DIR / "registration_session_registry_snapshot.csv",
        index=False,
    )